In [3]:
states = ['Rainy', 'Sunny']
observations = ['walk', 'shop', 'clean']

start_prob = {
    'Rainy': 0.6,
    'Sunny': 0.4
}

transition_prob = {
    'Rainy': {'Rainy': 0.7, 'Sunny': 0.3},
    'Sunny': {'Rainy': 0.4, 'Sunny': 0.6}
}

emission_prob = {
    'Rainy': {'walk': 0.1, 'shop': 0.4, 'clean': 0.5},
    'Sunny': {'walk': 0.6, 'shop': 0.3, 'clean': 0.1}
}

obs_sequence = ['walk', 'shop', 'clean']

def forward_algorithm(states, observations, start_prob, transition_prob, emission_prob, obs_seq):
    forward = []
    f0 = {}
    for state in states:
        f0[state] = start_prob[state] * emission_prob[state][obs_seq[0]]
    forward.append(f0)
    
    for t in range(1, len(obs_seq)):
        f_t = {}
        for curr_state in states:
            sum_prob = sum(forward[t-1][prev_state] * transition_prob[prev_state][curr_state] 
                           for prev_state in states)
            f_t[curr_state] = sum_prob * emission_prob[curr_state][obs_seq[t]]
        forward.append(f_t)
        
    total_prob = sum(forward[-1][state] for state in states)
    return forward, total_prob

steps, probability = forward_algorithm(states, observations, start_prob, transition_prob, emission_prob, obs_sequence)
print(f"Total Probability of sequence {obs_sequence}: {probability}")

Total Probability of sequence ['walk', 'shop', 'clean']: 0.033612


In [6]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
import pandas as pd
model=DiscreteBayesianNetwork([('B','A'),('T','A'),('E','A'),('E','T')])
cpt_b=TabularCPD(variable='B',variable_card=2,values=[[0.7],[0.3]])
cpt_e=TabularCPD(variable='E',variable_card=2,values=[[0.4],[0.6]])
cpt_t=TabularCPD(variable='T',variable_card=2,values=[[0.8,0.5],[0.2,0.5]],evidence=['E'],evidence_card=[2])
cpt_a=TabularCPD(variable='A',variable_card=2,values=[[1,0.9,0.95,0.85,0.89,0.7,0.87,0.3],[0,0.1,0.05,0.15,0.11,0.3,0.13,0.7]],evidence=['E','B','T'],evidence_card=[2,2,2])
model.add_cpds(cpt_b,cpt_e,cpt_a,cpt_t)
infer=VariableElimination(model)
result=infer.query(variables=['E'],evidence={'A':0})
print(result)


+------+----------+
| E    |   phi(E) |
+======+==========+
| E(0) |   0.4678 |
+------+----------+
| E(1) |   0.5322 |
+------+----------+


In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
import pandas as pd
model=DiscreteBayesianNetwork([('S','G'),('R','G'),('R','S')])
cpt_r=TabularCPD(variable='R',variable_card=2,values=[[0.2],[0.8]])
cpt_s=TabularCPD(variable='S',variable_card=2,values=[[0.01,0.4],[0.99,0.6]],evidence=['R'],evidence_card=[2])
cpt_g=TabularCPD(variable='G',variable_card=2,values=[[0,0.8,0.9,0.99],[1,0.2,0.1,0.01]],evidence=['S','R'],evidence_card=[2,2])
model.add_cpds(cpt_r,cpt_s,cpt_g)
infer=VariableElimination(model)
result=infer.query(variables=['R'],evidence={'G':0})
print(result)

   

+------+----------+
| R    |   phi(R) |
+======+==========+
| R(0) |   0.1960 |
+------+----------+
| R(1) |   0.8040 |
+------+----------+
